In [ ]:
# Common imports + DES (Modelica) translator pieces.
import json
import re
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from buildingspy.io.outputfile import Reader
from geojson_modelica_translator.modelica.GMT_Lib.DHC.DHC_5G_WH_GHX_HPTrio_VariableDist import (
    DHC5GWasteHeatGHXwithHPTrioVariableDist,
)
from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner
from geojson_modelica_translator.system_parameters.system_parameters import (
    SystemParameters,
)
from urbanopt_des.urbanopt_analysis import (
    URBANoptAnalysis as _UOA,
)  # local alias for clarity
from urbanopt_des.urbanopt_geojson import DESGeoJSON as URBANoptGeoJSON

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
)

# -- Execution mode -----------------------------------------------------------
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# This notebook now targets the shared coincident project directory directly.
MODEL_OUTPUT_SUBDIR = "../coincident"
SCENARIO_NAME = "classproject_optimized"
DES_MODEL_NAME = "five_g_agg_simple"

paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

### Activity 04c: TEN [Simple] 5G Scenario

In [ ]:
# Use the existing coincident project directly from the configured root path.
coincident_dir = paths.analysis_dir
uo_coincident = UO(coincident_dir.parent, "coincident", template_dir=template_data_dir)

# Weather data is already present in the coincident project. No
# need to update the weather information.

In [ ]:
# Common paths used by the DES (Modelica) steps below.
# Saved as relative-style paths because they're consumed from within Docker.
scenario_path = uo_coincident.project_path / f"{SCENARIO_NAME}.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# For Python-based DES creation with the TEN template, there isn't a need to create
# a _loop_order file.

### 5G: One-Week June Run (simple, no annual metrics yet)

In [ ]:
# 5G aggregated model — aggregate every URBANopt building load into a single
# "all_buildings" feature, then build / run the GMT 5G template against it.

uo_analysis_baseline_dir = uo_coincident.project_path
uo_analysis_baseline_scenario_name = SCENARIO_NAME
uo_analysis_baseline_results = (
    uo_analysis_baseline_dir / "run" / uo_analysis_baseline_scenario_name
)
project_geojson_filename = feature_path
# Create TEN files directly in the coincident project folder.
des_analysis_agg_dir = uo_coincident.project_path

uo_analysis = _UOA(project_geojson_filename, des_analysis_agg_dir)
uo_analysis.add_urbanopt_results(
    uo_analysis_baseline_dir, uo_analysis_baseline_scenario_name
)
uo_analysis.urbanopt.process_load_results(uo_analysis.geojson.get_building_ids())
uo_analysis.urbanopt.save_dataframes()
uo_analysis.urbanopt.display_name = "Non-Connected"

# 1. Aggregated DES project to roll up loads.
uo_analysis.urbanopt.create_abstract_run(
    "all_buildings", uo_analysis.urbanopt.data_loads
)

# 2. Create dedicated aggregated input files to avoid mutating originals.
new_geojson_file = des_analysis_agg_dir / f"{project_geojson_filename.stem}_agg.json"
shutil.copy(project_geojson_filename, new_geojson_file)

# 3. Update the GeoJSON to a single aggregated feature.
agg_geojson = URBANoptGeoJSON(new_geojson_file)
gdf_json = agg_geojson.create_aggregated_representation(["all"])
new_geojson_file.write_text(json.dumps(gdf_json, indent=2))

# 4. Update the scenario CSV so it has a single "All Buildings" row.
new_scenario_file = des_analysis_agg_dir / f"{scenario_path.stem}_agg.csv"
shutil.copy(scenario_path, new_scenario_file)
scenario_df = pd.read_csv(new_scenario_file)
scenario_df = scenario_df.head(1)
scenario_df["Feature Id"] = "all_buildings"
scenario_df["Feature Name"] = "All Buildings"
scenario_df.to_csv(new_scenario_file, index=False)

# 5. Generate system parameters for the aggregated rep.
# Use absolute paths in sys params to avoid cwd-dependent path resolution.
system_parameter_file = (
    des_analysis_agg_dir / f"{new_geojson_file.stem}_system_params.json"
)
system_parameter = SystemParameters()
system_parameter.csv_to_sys_param(
    model_type="time_series",
    sys_param_filename=system_parameter_file,
    scenario_dir=uo_analysis_baseline_results,
    feature_file=new_geojson_file,
    district_type="5G",
    relative_path=None,
    modelica_load_filename="modelica.mos",
    skip_weather_download=True,
)

model = DHC5GWasteHeatGHXwithHPTrioVariableDist(system_parameter)
des_analysis_baseline_dir = des_analysis_agg_dir / DES_MODEL_NAME
if des_analysis_baseline_dir.exists():
    shutil.rmtree(des_analysis_baseline_dir)

model.build_from_template(des_analysis_agg_dir, DES_MODEL_NAME)
(des_analysis_baseline_dir / "analysis_name.txt").write_text(DES_MODEL_NAME)
print("Created 5G aggregated model")

In [ ]:
# Run the generated 5G aggregated template model for one summer week in June.
if not (des_analysis_baseline_dir / "package.mo").exists():
    raise FileNotFoundError(
        f"Expected generated 5G package not found: {des_analysis_baseline_dir / 'package.mo'}"
    )

print(f"{DES_MODEL_NAME}.Districts.district")
print(f"file_to_load={des_analysis_baseline_dir / 'package.mo'}")
print(f"run_path={des_analysis_baseline_dir}")

# June 1st (non-leap year) to June 8th, in seconds from Jan 1.
summer_start_time = 151 * 24 * 3600
summer_stop_time = summer_start_time + 7 * 24 * 3600

results_path = (
    des_analysis_baseline_dir / f"{DES_MODEL_NAME}.Districts.district_results"
)
if results_path.exists():
    print("Results already exist, skipping model run.")
else:
    mr = ModelicaRunner()
    success, results_path = mr.run_in_docker(
        "compile_and_run",
        f"{DES_MODEL_NAME}.Districts.district",
        file_to_load=f"{des_analysis_baseline_dir / 'package.mo'}",
        run_path=des_analysis_baseline_dir,
        step_size=300,  # 5-min step
        start_time=summer_start_time,
        stop_time=summer_stop_time,
    )
    print(f"Run success: {success}")

print(f"5G results path: {results_path}")

In [ ]:
# Plot simple one-week results: loop temperatures + a few power/energy signals.
mat_path = (
    des_analysis_baseline_dir
    / f"{DES_MODEL_NAME}.Districts.district_results"
    / f"{DES_MODEL_NAME}.Districts.district_res.mat"
)
if not mat_path.exists():
    raise FileNotFoundError(f"MAT file not found: {mat_path}")

reader = Reader(str(mat_path), "dymola")
var_names = reader.varNames()

# Temperature plot: first available building supply/return pairs.
sup_re = re.compile(r"^dis\.con\[(\d+)\]\.junConSup\.vol\.T$")
ret_re = re.compile(r"^dis\.con\[(\d+)\]\.junConRet\.vol\.T$")
sup_ids = {int(m.group(1)) for v in var_names if (m := sup_re.match(v))}
ret_ids = {int(m.group(1)) for v in var_names if (m := ret_re.match(v))}
building_ids = sorted(sup_ids & ret_ids)

if not building_ids:
    raise ValueError(
        "No dis.con[*] supply/return temperature variables found in MAT file"
    )

plot_ids = building_ids[: min(4, len(building_ids))]
first_sup = f"dis.con[{plot_ids[0]}].junConSup.vol.T"
time_s, _ = reader.values(first_sup)
time_h = (time_s - time_s[0]) / 3600.0

plt.figure(figsize=(11, 5))
for bid in plot_ids:
    _, sup = reader.values(f"dis.con[{bid}].junConSup.vol.T")
    _, ret = reader.values(f"dis.con[{bid}].junConRet.vol.T")
    plt.plot(time_h, sup - 273.15, label=f"B{bid} supply")
    plt.plot(time_h, ret - 273.15, "--", label=f"B{bid} return")

plt.title("One-Week June Loop Temperatures")
plt.xlabel("Hours from run start")
plt.ylabel("Temperature [C]")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# Power/energy plot: find a few power-like variables and summarize MWh over the week.
power_candidates = [
    v
    for v in var_names
    if re.search(r"(^|\.)P($|\.)", v)
    and not re.search(r"port|ports|weaBus", v, flags=re.IGNORECASE)
]

if not power_candidates:
    print("No power-like variables found with pattern '...P'.")
else:
    # Pick a handful with the largest absolute peak to keep the plot readable.
    ranked = []
    for v in power_candidates[:150]:
        _, y = reader.values(v)
        ranked.append((float(np.max(np.abs(y))), v))
    ranked.sort(reverse=True)
    chosen = [v for _, v in ranked[: min(4, len(ranked))]]

    plt.figure(figsize=(11, 4.5))
    for v in chosen:
        _, y = reader.values(v)
        plt.plot(time_h, y / 1000.0, label=v)
    plt.title("One-Week June Power Signals")
    plt.xlabel("Hours from run start")
    plt.ylabel("Power [kW]")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    print("\nApproximate one-week energy by signal (integral of power):")
    for v in chosen:
        _, y = reader.values(v)
        # J = W*s, so divide by 3.6e9 for MWh.
        mwh = np.trapz(y, time_s) / 3.6e9
        print(f"  {v}: {mwh:.3f} MWh")